# FedFlower Phase 2 — Federated Learning

> **Before running:** Go to `Runtime → Change Runtime Type → T4 GPU → Save`

This notebook:
- Warm-starts from `best_model.pth` (upload when prompted)
- Runs a **5-client** FedAvg simulation (408 imgs/client, 10 rounds)
- Runs a **10-client** FedAvg simulation (204 imgs/client, 10 rounds)
- Tracks cross-entropy loss **per client per round**
- Reports per-client accuracy on local data and global test set
- Reports per-client macro F1 on global test set
- Produces `federated_accuracy.png`, `federated_loss.png`, `federated_results.json`

## Cell 1 — Install & Imports

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install scikit-learn matplotlib numpy -q
print('✅ Libraries ready')

import torch, torch.nn as nn, torch.optim as optim
import torchvision.datasets as datasets, torchvision.transforms as transforms, torchvision.models as models
from torch.utils.data import DataLoader, Subset, ConcatDataset
import numpy as np, matplotlib.pyplot as plt, copy, json, os
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if not torch.cuda.is_available():
    print('⚠️  No GPU — go to Runtime → Change Runtime Type → T4 GPU')

## Cell 2 — Upload best_model.pth

When the file picker appears, select `best_model.pth` from your laptop (downloaded after Phase 1).

In [ ]:
from google.colab import files
print('Upload best_model.pth from your laptop...')
uploaded = files.upload()
assert 'best_model.pth' in uploaded, '❌ Wrong filename — must be best_model.pth'
print(f'✅ Uploaded best_model.pth ({os.path.getsize("best_model.pth")/1e6:.1f} MB)')

## Cell 3 — Define FlowerCNN & Load Dataset

In [ ]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        self.backbone = models.resnet50(weights='IMAGENET1K_V2')
        for name, param in self.backbone.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                param.requires_grad = False
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4), nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.backbone(x)

# Strong augmentation for federated training data (per CONTEXT.md)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Clean transform for val/test — must match inference preprocessing exactly
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.Flowers102('./data', split='train', download=True, transform=train_transform)
val_data   = datasets.Flowers102('./data', split='val',   download=True, transform=train_transform)
test_data  = datasets.Flowers102('./data', split='test',  download=True, transform=test_transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')
print(f'   Federated pool: {len(train_data)+len(val_data)} images (train+val)')

## Cell 4 — Helper Functions

`local_train` returns **(state_dict, avg_loss)** so we can track cross-entropy per client per round.

In [ ]:
def local_train(global_model, loader, local_epochs=3, lr=5e-5, device='cuda'):
    """Train a local copy; return (weights, avg_cross_entropy_loss)."""
    local_model = copy.deepcopy(global_model)
    local_model.train()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, local_model.parameters()), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    total_loss, total_batches = 0.0, 0
    for _ in range(local_epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(local_model(imgs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            total_batches += 1
    avg_loss = total_loss / total_batches if total_batches > 0 else 0.0
    return local_model.state_dict(), avg_loss


def fedavg(global_model, client_weights_list, client_sizes):
    """Weighted average of client weights by dataset size."""
    total = sum(client_sizes)
    avg_w = copy.deepcopy(client_weights_list[0])
    for key in avg_w:
        avg_w[key] = torch.zeros_like(avg_w[key], dtype=torch.float32)
        for i, cw in enumerate(client_weights_list):
            avg_w[key] += cw[key].float() * (client_sizes[i] / total)
    global_model.load_state_dict(avg_w)
    return global_model


def evaluate_accuracy(model, loader, device):
    """Return Top-1 accuracy (%) on a DataLoader."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, preds = model(imgs).max(1)
            correct += preds.eq(labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total


def evaluate_accuracy_and_f1(model, loader, device):
    """Return (accuracy %, macro F1) on a DataLoader."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, preds = model(imgs).max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = 100.0 * np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
    f1  = f1_score(all_labels, all_preds, average='macro')
    return acc, f1


print('✅ local_train, fedavg, evaluate_accuracy, evaluate_accuracy_and_f1 defined')

## Cell 5 — Run 5-Client Federated Training (10 Rounds)

Each of the 5 clients gets ~408 images. Expected time: ~50 min.

In [ ]:
NUM_CLIENTS_5 = 5
GLOBAL_ROUNDS = 10
LOCAL_EPOCHS  = 3

all_data = ConcatDataset([train_data, val_data])  # 2040 images
np.random.seed(42)
indices_5 = np.random.permutation(len(all_data))
client_indices_5 = np.array_split(indices_5, NUM_CLIENTS_5)

client_loaders_5 = []
for i, idx in enumerate(client_indices_5):
    loader = DataLoader(Subset(all_data, idx.tolist()), batch_size=32, shuffle=True, num_workers=2)
    client_loaders_5.append(loader)
    print(f'  Client {i+1}: {len(idx)} images')

# Warm-start from centralized checkpoint
global_model_5 = FlowerCNN(num_classes=102).to(device)
global_model_5.load_state_dict(torch.load('best_model.pth', map_location=device))
print('✅ Warm-started from best_model.pth')

history_5 = {'round_acc': [], 'round_loss_per_client': [], 'round_loss_avg': []}

print(f'\nRunning {GLOBAL_ROUNDS} rounds × {NUM_CLIENTS_5} clients × {LOCAL_EPOCHS} local epochs')
print('=' * 65)
print(f'{"Round":>6}  {"C1 Loss":>9} {"C2 Loss":>9} {"C3 Loss":>9} {"C4 Loss":>9} {"C5 Loss":>9}  {"AvgLoss":>9}  {"TestAcc":>8}')
print('-' * 95)

for rnd in range(GLOBAL_ROUNDS):
    client_weights, client_sizes, client_losses = [], [], []
    for i, loader in enumerate(client_loaders_5):
        w, loss = local_train(global_model_5, loader, LOCAL_EPOCHS, lr=5e-5, device=device)
        client_weights.append(w)
        client_sizes.append(len(loader.dataset))
        client_losses.append(loss)

    global_model_5 = fedavg(global_model_5, client_weights, client_sizes)
    acc = evaluate_accuracy(global_model_5, test_loader, device)
    avg_loss = np.mean(client_losses)

    history_5['round_acc'].append(acc)
    history_5['round_loss_per_client'].append(client_losses)
    history_5['round_loss_avg'].append(avg_loss)

    loss_str = '  '.join(f'{l:.4f}' for l in client_losses)
    print(f'{rnd+1:>6}  {loss_str}  {avg_loss:>9.4f}  {acc:>7.2f}%')

print(f'\n✅ 5-client run complete. Best: {max(history_5["round_acc"]):.2f}%  Final: {history_5["round_acc"][-1]:.2f}%')

## Cell 6 — Run 10-Client Federated Training (10 Rounds)

Each of the 10 clients gets ~204 images. Expected time: ~50 min.

In [ ]:
NUM_CLIENTS_10 = 10

np.random.seed(42)
indices_10 = np.random.permutation(len(all_data))
client_indices_10 = np.array_split(indices_10, NUM_CLIENTS_10)

client_loaders_10 = []
for i, idx in enumerate(client_indices_10):
    loader = DataLoader(Subset(all_data, idx.tolist()), batch_size=32, shuffle=True, num_workers=2)
    client_loaders_10.append(loader)
    print(f'  Client {i+1:>2}: {len(idx)} images')

# Warm-start from centralized checkpoint (fresh copy — independent run)
global_model_10 = FlowerCNN(num_classes=102).to(device)
global_model_10.load_state_dict(torch.load('best_model.pth', map_location=device))
print('✅ Warm-started from best_model.pth')

history_10 = {'round_acc': [], 'round_loss_per_client': [], 'round_loss_avg': []}

print(f'\nRunning {GLOBAL_ROUNDS} rounds × {NUM_CLIENTS_10} clients × {LOCAL_EPOCHS} local epochs')
print('=' * 80)

for rnd in range(GLOBAL_ROUNDS):
    client_weights, client_sizes, client_losses = [], [], []
    for i, loader in enumerate(client_loaders_10):
        w, loss = local_train(global_model_10, loader, LOCAL_EPOCHS, lr=5e-5, device=device)
        client_weights.append(w)
        client_sizes.append(len(loader.dataset))
        client_losses.append(loss)

    global_model_10 = fedavg(global_model_10, client_weights, client_sizes)
    acc = evaluate_accuracy(global_model_10, test_loader, device)
    avg_loss = np.mean(client_losses)

    history_10['round_acc'].append(acc)
    history_10['round_loss_per_client'].append(client_losses)
    history_10['round_loss_avg'].append(avg_loss)

    per_client = '  '.join(f'{l:.4f}' for l in client_losses)
    print(f'Round {rnd+1:>2} | avg_loss={avg_loss:.4f} | test_acc={acc:.2f}%')
    print(f'         per-client: {per_client}')

print(f'\n✅ 10-client run complete. Best: {max(history_10["round_acc"]):.2f}%  Final: {history_10["round_acc"][-1]:.2f}%')

## Cell 7 — Per-Client Accuracy & F1 After Training

For the **5-client** model: each client is evaluated on (a) their own local data and (b) the full 6,149-image test set. Macro F1 is reported on the global test set.

In [ ]:
print('=' * 65)
print('PER-CLIENT EVALUATION — 5-CLIENT MODEL (after 10 rounds)')
print('=' * 65)
print(f'{"Client":>8}  {"Local Acc":>10}  {"Global Acc":>11}  {"Global F1":>10}')
print('-' * 55)

client_results_5 = []
for i, (loader, idx) in enumerate(zip(client_loaders_5, client_indices_5)):
    local_acc  = evaluate_accuracy(global_model_5, loader, device)
    global_acc, global_f1 = evaluate_accuracy_and_f1(global_model_5, test_loader, device)
    client_results_5.append({
        'client': i + 1,
        'local_acc': round(local_acc, 4),
        'global_acc': round(global_acc, 4),
        'global_f1': round(global_f1, 4),
        'n_images': len(idx)
    })
    print(f'{i+1:>8}  {local_acc:>9.2f}%  {global_acc:>10.2f}%  {global_f1:>10.4f}')

print()
print('=' * 65)
print('PER-CLIENT EVALUATION — 10-CLIENT MODEL (after 10 rounds)')
print('=' * 65)
print(f'{"Client":>8}  {"Local Acc":>10}  {"Global Acc":>11}  {"Global F1":>10}')
print('-' * 55)

client_results_10 = []
for i, (loader, idx) in enumerate(zip(client_loaders_10, client_indices_10)):
    local_acc  = evaluate_accuracy(global_model_10, loader, device)
    global_acc, global_f1 = evaluate_accuracy_and_f1(global_model_10, test_loader, device)
    client_results_10.append({
        'client': i + 1,
        'local_acc': round(local_acc, 4),
        'global_acc': round(global_acc, 4),
        'global_f1': round(global_f1, 4),
        'n_images': len(idx)
    })
    print(f'{i+1:>8}  {local_acc:>9.2f}%  {global_acc:>10.2f}%  {global_f1:>10.4f}')

## Cell 8 — Plot 1: Federated Accuracy vs Rounds

5-client line, 10-client line, and centralized 89.04% dashed baseline.

In [ ]:
CENTRALIZED_ACC = 89.04  # confirmed result from Phase 1
rounds = range(1, GLOBAL_ROUNDS + 1)

plt.figure(figsize=(11, 6))
plt.plot(rounds, history_5['round_acc'],  marker='o', color='#E63946',
         linewidth=2, markersize=8, label=f'5-client FedAvg (final: {history_5["round_acc"][-1]:.2f}%)')
plt.plot(rounds, history_10['round_acc'], marker='s', color='#2D9CDB',
         linewidth=2, markersize=8, label=f'10-client FedAvg (final: {history_10["round_acc"][-1]:.2f}%)')
plt.axhline(y=CENTRALIZED_ACC, color='#1F4E79', linewidth=2, linestyle='--',
            label=f'Centralized CNN ({CENTRALIZED_ACC}%)')

plt.xlabel('Federated Round', fontsize=12)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('Federated vs Centralized — Oxford 102 Flowers', fontweight='bold', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(rounds)
plt.tight_layout()
plt.savefig('federated_accuracy.png', dpi=150)
plt.show()
print('✅ Saved federated_accuracy.png')

## Cell 9 — Plot 2: Cross-Entropy Loss per Round

In [ ]:
plt.figure(figsize=(11, 6))
plt.plot(rounds, history_5['round_loss_avg'],  marker='o', color='#E63946',
         linewidth=2, markersize=8, label='5-client avg loss')
plt.plot(rounds, history_10['round_loss_avg'], marker='s', color='#2D9CDB',
         linewidth=2, markersize=8, label='10-client avg loss')

plt.xlabel('Federated Round', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.title('Federated Training Loss per Round — Oxford 102 Flowers', fontweight='bold', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(rounds)
plt.tight_layout()
plt.savefig('federated_loss.png', dpi=150)
plt.show()
print('✅ Saved federated_loss.png')

## Cell 10 — Save federated_results.json & Download All Files

Download these three files and keep them alongside `best_model.pth` on your laptop.

In [ ]:
results = {
    'centralized_acc': CENTRALIZED_ACC,
    '5_client': {
        'round_accuracies': history_5['round_acc'],
        'round_loss_avg':   history_5['round_loss_avg'],
        'round_loss_per_client': history_5['round_loss_per_client'],
        'per_client_eval': client_results_5,
        'final_acc': history_5['round_acc'][-1],
        'best_acc':  max(history_5['round_acc'])
    },
    '10_client': {
        'round_accuracies': history_10['round_acc'],
        'round_loss_avg':   history_10['round_loss_avg'],
        'round_loss_per_client': history_10['round_loss_per_client'],
        'per_client_eval': client_results_10,
        'final_acc': history_10['round_acc'][-1],
        'best_acc':  max(history_10['round_acc'])
    }
}

with open('federated_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('✅ Saved federated_results.json')

from google.colab import files
print('\nDownloading files...')
files.download('federated_results.json')
files.download('federated_accuracy.png')
files.download('federated_loss.png')
print('✅ All three files downloaded')